# conv-channel-sum — faded example 1: Compute one output pixel by summing over IC, KH, KW

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-channel-sum`. The last cell reports your progress on the `CNN: Channel-axis sum semantics` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Channel-axis sum semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-channel-sum`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-channel-sum"
DD_SUBTOPIC = "CNN: Channel-axis sum semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

At a single output position, conv2d takes the input patch of shape `(IC, KH, KW)`, multiplies it elementwise by one OC filter of shape `(IC, KH, KW)`, and sums **all** of those products into one scalar. The sum runs over the in-channel axis as well as the two kernel spatial axes — IC is contracted away.

## Faded exercise 1

### Faded — manual single-position channel sum

Implement `conv_one_position(x, weight, oc, oh, ow)` that returns the **scalar** `F.conv2d(x, weight)[0, oc, oh, ow]` for batch index 0, computed by hand. Slice the `(IC, KH, KW)` receptive field out of `x` at `(oh, ow)`, multiply by `weight[oc]`, and sum every element. The surrounding slicing is given; complete the elementwise-multiply-and-sum step.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn.functional as F

def conv_one_position(x, weight, oc, oh, ow):
    OC, IC, KH, KW = weight.shape
    patch = x[0, :, oh:oh+KH, ow:ow+KW]   # (IC, KH, KW)
    filt = weight[oc]                     # (IC, KH, KW)
    val = (patch * filt).sum()
    return val

t.manual_seed(0)
x = t.randn(1, 3, 6, 6)
weight = t.randn(4, 3, 3, 3)
print(conv_one_position(x, weight, 2, 1, 0))


def _test():
    import torch.nn.functional as F
    t.manual_seed(0)
    x = t.randn(1, 3, 6, 6)
    weight = t.randn(4, 3, 3, 3)
    ref = F.conv2d(x, weight)
    for oc in range(weight.shape[0]):
        for oh in range(ref.shape[2]):
            for ow in range(ref.shape[3]):
                got = conv_one_position(x, weight, oc, oh, ow)
                assert t.allclose(got, ref[0, oc, oh, ow], atol=1e-4), (oc, oh, ow, got, ref[0, oc, oh, ow])


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def conv_one_position(x, weight, oc, oh, ow):
    OC, IC, KH, KW = weight.shape
    patch = x[0, :, oh:oh+KH, ow:ow+KW]   # (IC, KH, KW)
    filt = weight[oc]                     # (IC, KH, KW)
    val = (patch * filt).sum()
    return val

t.manual_seed(0)
x = t.randn(1, 3, 6, 6)
weight = t.randn(4, 3, 3, 3)
print(conv_one_position(x, weight, 2, 1, 0))
```
</details>